In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/small_bench/checkpoints/pre_cell_22.pickle

In [ ]:
%%cudf.pandas.profile
### cell 22 ###
def get_n_grams(n_grams, top_n=10):
    # Revert to the original CountVectorizer-based implementation
    df_words = pd.DataFrame()
    for dt in tqdm(train['discourse_type'].unique()):
        df = train.query(f"discourse_type == '{dt}'")
        texts = df['discourse_text'].tolist()
        vec = CountVectorizer(lowercase=True,
                              stop_words='english',
                              ngram_range=(n_grams, n_grams)).fit(texts)
        bag_of_words = vec.transform(texts)
        sum_words = bag_of_words.sum(axis=0)
        words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
        cvec_df = pd.DataFrame.from_records(
            words_freq,
            columns=['words', 'counts']
        ).sort_values(by='counts', ascending=False)
        cvec_df.insert(0, 'Discourse_type', dt)
        cvec_df = cvec_df.iloc[:top_n, :]
        df_words = df_words._append(cvec_df)
    return df_words

# compute bigrams
bigrams = get_n_grams(n_grams=2, top_n=10)
bigrams.head()